In [0]:
from pyspark.sql import functions as F

# ==========================================================
# Configuration
# ==========================================================

SILVER_TABLE = "finance_intelligence_hub.silver.company_news_polygon"
GOLD_VIEW = "finance_intelligence_hub.gold.fact_news"

print("="*80)
print("Company News Gold View")
print("="*80)

# ==========================================================
# Create / Replace Gold View
# A VIEW (not a table) so it always reflects the latest silver data with
# zero duplication of storage, and re-running this is always safe —
# CREATE OR REPLACE never touches or drops the underlying silver data.
# ==========================================================

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_VIEW} AS
SELECT
    article_id,
    ticker,
    related_tickers,
    related_tickers_count,
    title,
    description,
    text_for_finbert,
    text_word_count,
    author_name,
    author_email,
    publisher_name,
    publisher_url,
    source_url,
    image_url,
    image_url_is_valid,
    published_at
FROM {SILVER_TABLE}
""")

print(f"Gold view {GOLD_VIEW} created/refreshed successfully.")

# ==========================================================
# Quick sanity check
# ==========================================================

row_count = spark.sql(f"SELECT COUNT(*) FROM {GOLD_VIEW}").first()[0]
print(f"Gold view row count: {row_count:,}")

spark.sql(f"DESCRIBE {GOLD_VIEW}").show(truncate=False)